# 01 — Data Acquisition & Cleaning

This notebook demonstrates the reproducible pipeline for retrieving and cleaning
NASA GLOBE Observer Clouds 2022 Challenge data.

**Data source:** [GLOBE Observer](https://observer.globe.gov/get-data)  
**Protocol:** [GLOBE Clouds](https://www.globe.gov/web/atmosphere/protocols/clouds)  
**Citation:** Global Learning and Observations to Benefit the Environment (GLOBE) Program, 2024, globe.gov

## Overview

1. Fetch raw cloud observations from the GLOBE API (with caching)
2. Inspect raw data quality
3. Clean and filter to produce a tidy Parquet dataset
4. Document provenance and schema

In [ ]:
import sys
from pathlib import Path

# Ensure the package is importable
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

from globe_cloud_insights.config import RAW_DIR, PROCESSED_DIR, GLOBE_START_DATE, GLOBE_END_DATE
from globe_cloud_insights.fetch import fetch_globe_data, build_api_url
from globe_cloud_insights.clean import clean_data, get_schema_markdown

## Step 1: Data Acquisition

The GLOBE API provides programmatic access to citizen-science observations.
We query the `sky_conditions` protocol for the challenge period
(Jan 15 – Feb 15, 2022). The pipeline:

- Splits the date range into weekly chunks (staying under the 1M row API limit)
- Caches the raw CSV to `data/raw/` to avoid redundant downloads
- Returns a combined DataFrame

In [ ]:
# Show the API URL we'll be calling
print('API URL (sample):', build_api_url(sample=True))
print(f'Date range: {GLOBE_START_DATE} to {GLOBE_END_DATE}')

In [ ]:
# Fetch data (uses cache if available)
df_raw = fetch_globe_data()
print(f'Raw records: {len(df_raw):,}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

## Step 2: Raw Data Inspection

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum())

## Step 3: Cleaning Pipeline

The cleaning pipeline:
- Renames columns to a consistent schema
- Parses datetime fields
- Drops rows with missing or out-of-range coordinates
- Drops rows without timestamps
- Computes cloud cover percentage from sky condition labels
- Assigns observation IDs where missing
- Saves to Parquet format

In [ ]:
df_clean, provenance = clean_data()
print(f'Clean records: {len(df_clean):,}')
print(f'\nProvenance:')
for k, v in provenance.items():
    print(f'  {k}: {v}')

In [ ]:
df_clean.head(10)

## Step 4: Schema Documentation

The cleaned dataset follows this schema:

In [ ]:
from IPython.display import Markdown
Markdown(get_schema_markdown())

In [ ]:
print(f'\nDataset saved to: {PROCESSED_DIR / "globe_clouds_2022_clean.parquet"}')
print(f'Schema columns: {list(df_clean.columns)}')
print(f'\nData types:\n{df_clean.dtypes}')